# SERAFIM

In [36]:
from sentence_transformers import SentenceTransformer
import numpy as np

sentences = ["Olá", "Olá"]

model = SentenceTransformer('PORTULAN/serafim-900m-portuguese-pt-sentence-encoder-ir')
embeddings = model.encode(sentences)
print(embeddings)


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

[[-1.600383   -0.16803083  0.15716419 ...  1.0583158  -1.8078195
  -0.69922423]
 [-1.600383   -0.16803083  0.15716419 ...  1.0583158  -1.8078195
  -0.69922423]]


In [52]:
sentences = ["Pipocas da russia", "O rato roeu a rolha da garrafa do rei da Russia"]

embeddings = model.encode(sentences)

emb0 = np.sqrt(np.inner(embeddings[0], embeddings[0]))
emb1 = np.sqrt(np.inner(embeddings[1], embeddings[1]))
print(emb0, emb1, np.inner(embeddings[0], embeddings[1]) / (emb0 * emb1))

31.80001 29.115202 0.5242693


# AMALIA

In [56]:
sentences = ["Estrada da beira", "Beira da estrada"]

embeddings = model.encode(sentences)

emb0 = np.sqrt(np.inner(embeddings[0], embeddings[0]))
emb1 = np.sqrt(np.inner(embeddings[1], embeddings[1]))
print(emb0, emb1, np.inner(embeddings[0], embeddings[1]))

32.17438 32.907463 896.9918


In [42]:
np.inner(embeddings[0], embeddings[0])

np.float32(1112.0796)

In [43]:
np.inner(embeddings[1], embeddings[1])

np.float32(1112.0796)

In [46]:
embeddings[0] * embeddings[0]

array([2.561226  , 0.02823436, 0.02470058, ..., 1.1200322 , 3.2682114 ,
       0.48891452], shape=(1536,), dtype=float32)

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

mid = "amalia-llm/AMALIA-9B-0626-DPO"
tok = AutoTokenizer.from_pretrained(mid)
model = AutoModelForCausalLM.from_pretrained(
    mid, dtype=torch.bfloat16, attn_implementation="eager"
).to("mps")

messages = [
    # optional; if omitted the template injects the default AMALIA system prompt
    {"role": "system", "content": "És a AMALIA, um assistente útil. Responde em português europeu."},
    {"role": "user", "content": "Explica a diferença entre 'à' e 'há'."},
]

Loading weights:   0%|          | 0/381 [00:00<?, ?it/s]

In [2]:
inputs = tok.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

In [ ]:
out = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True, temperature=0.7, top_p=0.95,
    eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"),
)
print(tok.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))